# Tree Ensembles: Random Forests & Boosting

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/konstantin-schekotihin/dl_course/blob/master/02_ML/05_ensembles.ipynb)


In [ ]:
# Setup & configuration (skip in presentation slides)
%matplotlib inline
import os
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_moons, load_breast_cancer
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split
import sklearn.metrics as mt

# Optional Weights & Biases experiment logging
try:
    import wandb
    WANDB_AVAILABLE = True
except ImportError:
    WANDB_AVAILABLE = False

sns.set_theme(
    style="whitegrid",
    palette="deep",
    rc={
        "figure.figsize": (8, 6),
        "font.size": 14,
        "axes.labelsize": 14,
        "xtick.labelsize": 12,
        "ytick.labelsize": 12,
    }
)


# Tree Ensembles: The Tabular Gold Standard

- While Deep Neural Networks dominate unstructured perception tasks (vision, audio, natural language), **Tree Ensembles** remain the undisputed champion for **structured, tabular data** (economics, healthcare, marketing, finance).
- Why do ensembles dominate tabular data?
  - **Robust to unnormalized features**: Invariant to monotonic feature scalings (no `StandardScaler` strictly required).
  - **Handles mixed data types**: Naturally ingests categorical indicators, discrete counts, and continuous variables.
  - **Captures complex feature interactions**: Hierarchical splits automatically discover non-linear synergies without manual polynomial feature engineering.
  - **High performance with minimal tuning**: Outstanding out-of-the-box predictive accuracy.


## Refresh: Decision Tree Induction & Splitting

- *Recap from Introduction to AI 2 (Russell & Norvig 4th ed., Ch. 19)*: A decision tree partitions feature space into axis-aligned hyper-rectangles via recursive greedy top-down induction (Hunt's Algorithm, CART, ID3/C4.5).
- At internal node $S$, the algorithm searches over all features $j$ and split thresholds $\theta$ to maximize **Information Gain** (or impurity drop):

$$
\Delta(S, j, \theta) = I(S) - \frac{|S_L|}{|S|} I(S_L) - \frac{|S_R|}{|S|} I(S_R)
$$

- Common impurity metrics $I(S)$ for class distribution $\{p_1, \dots, p_K\}$:
  - **Entropy**: $H(S) = -\sum_{k=1}^K p_k \log_2 p_k$
  - **Gini Impurity**: $I_G(S) = 1 - \sum_{k=1}^K p_k^2$
  - **Classification Error**: $E(S) = 1 - \max_k p_k$


### The Overfitting Dilemma of Single Decision Trees

- An unconstrained decision tree grows until every single leaf is completely pure ($I(S)=0$):
  - **Training Error**: Drops to $0\%$ (**low bias**).
  - **Generalization**: Severely degrades on test data (**high variance**).
- Single trees produce jagged, fragmented step boundaries that memorize idiosyncratic noise.
- **Controlling Tree Capacity**:
  - **Pre-pruning (Early Stopping)**: Restrict `max_depth`, require `min_samples_split`, or set `min_samples_leaf`.
  - **Post-pruning (Cost-Complexity)**: Minimize penalized loss $R_\alpha(T) = R(T) + \alpha |T|$, balancing empirical error $R(T)$ against tree complexity $|T|$.


In [ ]:
def plot_surface(model, X, y, ax, title=""):
    """Helper to render 2D classification decision surface."""
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 250), np.linspace(y_min, y_max, 250))
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    
    ax.contourf(xx, yy, Z, alpha=0.25, cmap=plt.cm.coolwarm)
    ax.scatter(X[y == 0, 0], X[y == 0, 1], c='crimson', label='Class 0', edgecolors='k', alpha=0.7)
    ax.scatter(X[y == 1, 0], X[y == 1, 1], c='dodgerblue', label='Class 1', edgecolors='k', alpha=0.7)
    ax.set_title(title)
    ax.set_xlabel("$x_1$")
    ax.set_ylabel("$x_2$")
    ax.legend(loc='lower left')


In [ ]:
# Generate noisy non-linear two moons dataset
X, y = make_moons(n_samples=400, noise=0.30, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# Fit constrained vs. unconstrained single decision trees
tree_pruned = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_tr, y_tr)
tree_deep = DecisionTreeClassifier(max_depth=None, random_state=42).fit(X_tr, y_tr)

acc_pruned_te = tree_pruned.score(X_te, y_te)
acc_deep_tr = tree_deep.score(X_tr, y_tr)
acc_deep_te = tree_deep.score(X_te, y_te)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_surface(tree_pruned, X_te, y_te, axes[0], title=f"Pruned Tree (max_depth=3) | Test: {acc_pruned_te:.1%}")
plot_surface(tree_deep, X_te, y_te, axes[1], title=f"Unconstrained Tree | Train: {acc_deep_tr:.0%}, Test: {acc_deep_te:.1%}")
plt.tight_layout()
plt.show()


## The Ensemble Philosophy: The Wisdom of Crowds

- **Condorcet's Jury Theorem (1785)**: If $B$ independent voters each have probability $p > 0.5$ of reaching a correct verdict, the probability that the majority vote is correct approaches $1.0$ as $B \to \infty$!
- In Machine Learning, an **ensemble** aggregates predictions from $B$ diverse base estimators:

$$
\hat{y}_{\mathrm{ens}}(\mathbf{x}) = \mathrm{mode}\left( \{ \hat{y}_b(\mathbf{x}) \}_{b=1}^B \right) \quad (\text{classification}), \qquad \hat{y}_{\mathrm{ens}}(\mathbf{x}) = \frac{1}{B} \sum_{b=1}^B f_b(\mathbf{x}) \quad (\text{regression})
$$

- **Two Fundamental Ensemble Paradigms**:
  1. **Bagging (Bootstrap Aggregation)**: Trains diverse high-variance models in **parallel** on randomized data subsets $\implies$ **slashes variance**.
  2. **Boosting**: Trains shallow high-bias models in **series**, where each tree learns to predict the residual errors of prior trees $\implies$ **slashes bias**.


## Bootstrap Aggregation: Bagging

- How do we obtain $B$ distinct training sets from a single empirical sample $\mathcal{D}$ of size $N$?
- **The Non-Parametric Bootstrap (Efron 1979)**: Sample $N$ instances *uniformly at random with replacement* from $\mathcal{D}$.
- What fraction of original observations are omitted from a bootstrap replicate?

$$
P(\text{sample } n \text{ omitted in } N \text{ draws}) = \left(1 - \frac{1}{N}\right)^N \xrightarrow{N \to \infty} \frac{1}{e} \approx 36.8\%
$$

- Each bootstrap dataset contains approximately **$63.2\%$ unique observations**.
- The remaining **$36.8\%$ unused observations** are called **Out-of-Bag (OOB)** samples!


## Random Forests: Decorrelating the Trees

- **Leo Breiman (2001)** identified a fundamental limitation of naive tree bagging:
  - If a few features are strong predictors, almost every bootstrap tree splits on those same dominant features near the root.
  - Consequently, individual tree predictions become **strongly correlated** ($r > 0$).
  - For $B$ estimators with variance $\sigma^2$ and pairwise correlation $\rho$, the variance of their average is:

$$
\mathrm{var}\left( \frac{1}{B} \sum_{b=1}^B f_b \right) = \rho \sigma^2 + \frac{1 - \rho}{B} \sigma^2 \xrightarrow{B \to \infty} \rho \sigma^2
$$

- **The Random Forest Solution (Feature Subsampling)**:
  - At *every candidate split*, sample only $m < D$ features at random (default $m = \lfloor \sqrt{D} \rfloor$).
  - Forces trees to explore diverse feature combinations, driving $\rho \to 0$ and drastically minimizing ensemble variance!


### Out-of-Bag (OOB) Evaluation & Feature Importance

- **Free Cross-Validation via Out-of-Bag (OOB) Score**:
  - For observation $(\mathbf{x}_n, y_n)$, predict using only the subset of trees (${\approx} B/3$) where $\mathbf{x}_n$ was omitted.
  - Aggregating OOB predictions yields an unbiased generalization estimate **without needing a holdout validation set**!
- **Mean Decrease in Impurity (MDI) Feature Importance**:
  - Quantifies how much feature $j$ contributes to reducing node impurity across all splits in all $B$ trees:

$$
\mathrm{MDI}(j) = \frac{1}{B} \sum_{b=1}^B \sum_{t \in T_b : v(t) = j} \frac{N_t}{N} \Delta I(t)
$$

- Automatically ranks the explanatory power of each predictor in the dataset.


In [ ]:
# Train Random Forest with Out-of-Bag scoring
rf = RandomForestClassifier(n_estimators=100, max_features='sqrt', oob_score=True, random_state=42)
rf.fit(X_tr, y_tr)

acc_rf_te = rf.score(X_te, y_te)
print(f"Random Forest Out-of-Bag (OOB) Accuracy : {rf.oob_score_:.1%}")
print(f"Random Forest Independent Test Accuracy  : {acc_rf_te:.1%}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_surface(tree_deep, X_te, y_te, axes[0], title=f"Single Deep Tree (Acc: {acc_deep_te:.1%})")
plot_surface(rf, X_te, y_te, axes[1], title=f"Random Forest (100 Trees | Acc: {acc_rf_te:.1%})")
plt.tight_layout()
plt.show()


## Boosting: The Sequential Correction Paradigm

- While Random Forests build independent trees in parallel, **Boosting** builds trees **sequentially in series**.
- Each new tree is specifically trained to correct the errors made by all previous trees combined:

$$
F_M(\mathbf{x}) = F_0(\mathbf{x}) + \sum_{m=1}^M \eta f_m(\mathbf{x})
$$

- **The Shrinkage Parameter $\eta \in (0, 1]$ (Learning Rate)**:
  - Scales the contribution of each newly added tree.
  - Lower $\eta$ (e.g. $0.05$ or $0.1$) acts as a regularizer, requiring more trees ($M$) but preventing individual trees from overfitting.
- Two fundamental milestones in boosting literature:
  1. **AdaBoost** (Freund & Schapire 1997): Reweights misclassified instances exponentially.
  2. **Gradient Boosting (GBM)** (Friedman 2001): Performs functional gradient descent on arbitrary loss functions.


### Gradient Boosted Decision Trees (Friedman 2001)

- Consider empirical risk minimization for differentiable loss $L(y, F(\mathbf{x}))$:

$$
\min_F \sum_{n=1}^N L(y_n, F(\mathbf{x}_n))
$$

- At stage $m$, we compute the **pseudo-residuals** (the negative gradient of loss with respect to current predictions):

$$
r_{nm} = - \left[ \frac{\partial L(y_n, F(\mathbf{x}_n))}{\partial F(\mathbf{x}_n)} \right]_{F(\mathbf{x}) = F_{m-1}(\mathbf{x})}
$$

- A shallow regression tree $h_m(\mathbf{x})$ is fit to targets $r_{nm}$ using squared error.
- Update model: $F_m(\mathbf{x}) = F_{m-1}(\mathbf{x}) + \eta h_m(\mathbf{x})$.
> 💡 **Intuition:** Successive trees step in the direction of steepest descent in function space!


In [ ]:
# Train Gradient Boosting Classifier
gb = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
gb.fit(X_tr, y_tr)
acc_gb_te = gb.score(X_te, y_te)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_surface(rf, X_te, y_te, axes[0], title=f"Random Forest (Bagging | Acc: {acc_rf_te:.1%})")
plot_surface(gb, X_te, y_te, axes[1], title=f"Gradient Boosting (Boosting | Acc: {acc_gb_te:.1%})")
plt.tight_layout()
plt.show()


In [ ]:
# Inspect error reduction across boosting iterations via staged_predict
test_errors_gb = [mt.zero_one_loss(y_te, y_pred) for y_pred in gb.staged_predict(X_te)]
train_errors_gb = [mt.zero_one_loss(y_tr, y_pred) for y_pred in gb.staged_predict(X_tr)]

plt.figure(figsize=(9, 4.5))
plt.plot(range(1, 101), train_errors_gb, label='Training Misclassification Error', color='royalblue', lw=2)
plt.plot(range(1, 101), test_errors_gb, label='Test Misclassification Error', color='crimson', lw=2)
plt.title("Gradient Boosting Test Error Across Iterations (Shrinkage $\\eta=0.1$)")
plt.xlabel("Number of Boosting Stages ($M$)")
plt.ylabel("Misclassification Error Rate")
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()


## Applied Case Study: Breast Cancer Diagnostic Benchmark

- **Dataset**: Wisconsin Diagnostic Breast Cancer dataset (569 patients, $D=30$ continuous nuclear features).
- **Task $(T, P, E)$**:
  - $T$: Predict whether a biopsied tissue mass is benign ($0$) or malignant ($1$).
  - $P$: Cross-validated Accuracy, $F_1$-score, and ROC-AUC.
  - $E$: 569 digitized cell nuclei measurement vectors.
- We compare three generations of tree-based architectures:
  1. **Single Decision Tree** (`max_depth=4`)
  2. **Random Forest** (`n_estimators=100`)
  3. **Gradient Boosted Decision Trees** (`n_estimators=100, learning_rate=0.1`)


In [ ]:
# Load breast cancer dataset
cancer = load_breast_cancer()
X_c, y_c = cancer.data, cancer.target
X_c_tr, X_c_te, y_c_tr, y_c_te = train_test_split(X_c, y_c, test_size=0.25, random_state=42, stratify=y_c)

models = {
    "Single Tree (depth=4)": DecisionTreeClassifier(max_depth=4, random_state=42),
    "Random Forest (100)": RandomForestClassifier(n_estimators=100, oob_score=True, random_state=42),
    "Gradient Boosting (100)": GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
}

records = []
for name, m in models.items():
    m.fit(X_c_tr, y_c_tr)
    preds = m.predict(X_c_te)
    probs = m.predict_proba(X_c_te)[:, 1]
    records.append({
        "Model": name,
        "Train Acc": f"{m.score(X_c_tr, y_c_tr):.2%}",
        "Test Acc": f"{mt.accuracy_score(y_c_te, preds):.2%}",
        "F1-Score": f"{mt.f1_score(y_c_te, preds):.3f}",
        "ROC-AUC": f"{mt.roc_auc_score(y_c_te, probs):.3f}"
    })

df_results = pd.DataFrame(records)
print("=== TABULAR MODEL COMPARISON: BREAST CANCER DIAGNOSTIC ===")
print(df_results.to_string(index=False))


In [ ]:
# Compare top 8 feature importances (MDI) across Random Forest and Gradient Boosting
m_rf = models["Random Forest (100)"]
m_gb = models["Gradient Boosting (100)"]

top_idx = np.argsort(m_rf.feature_importances_)[::-1][:8]
top_features = cancer.feature_names[top_idx]

fig, ax = plt.subplots(figsize=(10, 5))
bar_width = 0.35
y_pos = np.arange(len(top_features))

ax.barh(y_pos - bar_width/2, m_rf.feature_importances_[top_idx], bar_width, label='Random Forest (MDI)', color='navy', alpha=0.8)
ax.barh(y_pos + bar_width/2, m_gb.feature_importances_[top_idx], bar_width, label='Gradient Boosting (MDI)', color='darkorange', alpha=0.8)

ax.set_yticks(y_pos)
ax.set_yticklabels(top_features)
ax.invert_yaxis()
ax.set_xlabel("Relative Feature Importance (MDI)")
ax.set_title("Top 8 Diagnostic Predictors: Random Forest vs. Gradient Boosting")
ax.legend()
plt.tight_layout()
plt.show()


## Mandatory Course Tooling: Weights & Biases (W&B)

- As mandated by the course grading modalities ([`02_ML/modalities.md`](file:///Users/kostya/Documents/Teaching/ML-DL/02_ML/modalities.md#L30-L33)):
  - All student project teams must document model iteration progress and hyperparameter sweeps in **Weights & Biases**.
  - W&B ensures reproducible science, automatic metric tracking, and defensible comparison tables.
- Standard W&B Experiment Tracking Workflow:
  1. **Initialize Run**: `wandb.init(project="project-name", config=hyperparameters)`.
  2. **Log Metrics**: `wandb.log({"accuracy": acc, "f1_score": f1, "roc_auc": auc})`.
  3. **Log Diagnostic Visualizations**: `wandb.log({"roc_curve": wandb.plot.roc_curve(...)})`.
  4. **Close Run**: `wandb.finish()`.


In [ ]:
# W&B Project Sweep Logging (runs in offline/disabled mode when no API key is provided)
sweep_configs = [
    {"n_estimators": 25, "max_depth": 3, "learning_rate": 0.1},
    {"n_estimators": 50, "max_depth": 3, "learning_rate": 0.1},
    {"n_estimators": 100, "max_depth": 3, "learning_rate": 0.1},
    {"n_estimators": 100, "max_depth": 5, "learning_rate": 0.05}
]

sweep_results = []
for cfg in sweep_configs:
    # Initialize run (disabled mode for seamless offline execution without prompts)
    run = wandb.init(project="ensemble-benchmark", config=cfg, mode="disabled") if WANDB_AVAILABLE else None
    
    clf = GradientBoostingClassifier(**cfg, random_state=42)
    clf.fit(X_c_tr, y_c_tr)
    preds = clf.predict(X_c_te)
    acc = mt.accuracy_score(y_c_te, preds)
    f1 = mt.f1_score(y_c_te, preds)
    
    if run is not None:
        wandb.log({"test_acc": acc, "test_f1": f1})
        run.finish()
    
    sweep_results.append({**cfg, "Test Accuracy": f"{acc:.2%}", "Test F1": f"{f1:.3f}"})

print("=== W&B LOGGED HYPERPARAMETER SWEEP SUMMARY ===")
print(pd.DataFrame(sweep_results).to_string(index=False))


## Algorithm Selection Card: Tree Ensembles

| Dimension | Random Forests (Bagging) | Gradient Boosted Trees (Boosting) |
| :--- | :--- | :--- |
| **Core Mechanism** | Parallel independent trees; averages predictions to **reduce variance**. | Sequential weak trees; fits residuals of previous trees to **reduce bias**. |
| **Best Suited For** | Robust, general-purpose tabular classification & regression with zero tuning needed. | Maximum competitive predictive accuracy on complex tabular problems. |
| **Key Hyperparameters** | • `n_estimators`: number of trees (higher is always better; plateaus).<br>• `max_features`: subset size per split (default $\sqrt{D}$). | • `learning_rate` ($\eta$): step size shrinkage.<br>• `n_estimators`: number of boosting stages.<br>• `max_depth`: shallow trees (depth 3 to 6). |
| **Computational Scaling** | Fast and embarrassingly parallelizable across all CPU cores. | Sequential; can be slower to train, though optimized engines (HistGradientBoosting, XGBoost, LightGBM) accelerate computation. |
| **Critical Failure Modes** | Cannot extrapolate continuous values outside the range observed during training; high memory footprint for large ensembles. | Susceptible to overfitting if `learning_rate` is too large or `n_estimators` is untuned; sensitive to label noise. |

> 💡 **Key Takeaway:** Tree Ensembles are the gold standard for tabular data. Use **Random Forests** for fast, reliable, tuning-free baselines with OOB validation; use **Gradient Boosting** when squeezing out peak predictive performance.
